# Training Proposed Model — IndoBERT + CNN 1D (Dual-Path)

**Notebook:** `03_indobert_cnn_training.ipynb`  
**Peneliti:** Khoeru Roziqin  
**Tanggal:** 01-04-26  

---

## Arsitektur Dual-Path [CLS] + CNN

```
IndoBERT → last_hidden_state [batch, 128, 768]
                |
   ┌────────────┴────────────┐
   ▼                         ▼
Path 1: [CLS] Token    Path 2: CNN 1D Multi-Kernel
[batch, 768]           Conv1D(k1/k2/k3) → GlobalMaxPool
konteks global         [batch, filter×n_kernel]
Dropout(0.1)           pola n-gram lokal
   │                         │
   └──── Concatenate ─────────┘
        [batch, 768 + filter×n_kernel]
               ↓
        Dropout → Dense(256) → Dropout → Dense(3)
```

**[CLS]** = representasi konteks global kalimat (Devlin et al., 2019).  
**CNN** = pola n-gram lokal (Kim, 2014).  
**Concatenate** = CNN melengkapi [CLS], tidak menggantikannya (Chen et al., 2020).

**Referensi:** Devlin et al. (2019); Kim (2014); Chen et al. (2020); Howard & Ruder (2018); Loshchilov & Hutter (2019).

---

## Cara Penggunaan — 4 Sesi

| Sesi | Phase | Aksi setelah selesai |
|---|---|---|
| 1 | Phase 1 | Update `BEST_LR_BERT`, `BEST_LR_CNN`, `BEST_BS` |
| 2 | Phase 2A | Hasil ke `phase2A_results.csv` |
| 3 | Phase 2B | Update semua `BEST_*` CNN |
| 4 | Phase 3 | Semua output final tersimpan |

Sel **0–4 harus dijalankan ulang** di awal setiap sesi.

## 0. Setup

> **Jalankan di awal SETIAP sesi.**  
> Default: **Google Colab**. Untuk Kaggle: uncomment sel Kaggle, comment sel Colab.

In [ ]:
import sys, os

IS_KAGGLE = os.path.exists('/kaggle/working')
IS_COLAB  = 'google.colab' in sys.modules or os.path.exists('/content')

if IS_KAGGLE:
    print('Environment: Kaggle Notebook')
elif IS_COLAB:
    print('Environment: Google Colab')
else:
    print('Environment: Local / Unknown')

In [ ]:
# ══════════════════════════════════════════════════════
#  SETUP — KAGGLE NOTEBOOK
# ══════════════════════════════════════════════════════
import os, torch
def check_gpu_kaggle():
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA tidak tersedia. Aktifkan GPU di Settings.')
    try:
        t = torch.zeros(2,2).cuda(); _ = t @ t; del t; torch.cuda.empty_cache()
        cap = torch.cuda.get_device_capability()
        print(f'✅ GPU OK: {torch.cuda.get_device_name(0)} | sm_{cap[0]}{cap[1]}')
    except Exception as e:
        raise RuntimeError(f'GPU error: {e}\nSolusi: Restart session & realokasi GPU.')
check_gpu_kaggle()
print('✅ Setup Kaggle selesai.')

In [ ]:
import time, warnings, random, itertools
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.utils import resample
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device  : {DEVICE}')
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability()
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
    print(f'Compute : sm_{cap[0]}{cap[1]}')
print(f'PyTorch : {torch.__version__}')

## 1. Konfigurasi Global

> **Jalankan di awal SETIAP sesi.**

In [ ]:
# ── PATH — KAGGLE ────────
INPUT_PATH    = '/kaggle/input/datasets/khoeruroziqin/final-mbg-labeled/final_mbg_labeled.csv'
OUTPUT_DIR    = '/kaggle/working/model_results'
MODEL_DIR     = '/kaggle/working/saved_models'
OUTPUTS_INPUT = '/kaggle/input/datasets/khoeruroziqin/final-training-dual-path-outputs'

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR,  exist_ok=True)

# ── Restore file dari sesi sebelumnya (Kaggle only) ───────────────────────────
import shutil
_files_to_restore = [
    # ── Selalu ada di setiap sesi ─────────────────────────────────────────
    'test_set_indices.npy', 'test_set_fixed.csv',
    'phase1_results.csv', 'phase1_epoch_history.csv',
    'phase1_fold_details.csv', 'phase1_summary_top10.csv',
    # ── Tambahkan untuk sesi Phase 2B ─────────────────────────────────────
    'phase2A_results.csv', 'phase2A_epoch_history.csv', 'phase2A_fold_details.csv',
    # ── Tambahkan untuk sesi Phase 3 ──────────────────────────────────────
    'phase2B_results.csv', 'phase2B_epoch_history.csv', 'phase2B_fold_details.csv',
    'phase2_combined_results.csv', 'phase2_summary_top10.csv',
]
print('Restoring files...')
for fname in _files_to_restore:
    src = f'{OUTPUTS_INPUT}/{fname}'; dst = f'{OUTPUT_DIR}/{fname}'
    if os.path.exists(src): shutil.copy(src, dst); print(f'  ✔ {fname}')
    else: print(f'  ⚠️  Tidak ditemukan: {fname}')
from datetime import datetime
print('\nVerifikasi:')
for fname in _files_to_restore:
    dst = f'{OUTPUT_DIR}/{fname}'
    if os.path.exists(dst):
        size  = os.path.getsize(dst)
        mtime = datetime.fromtimestamp(os.path.getmtime(dst)).strftime('%Y-%m-%d %H:%M')
        info  = f'{size/1024:.1f} KB | {mtime}'
        if fname.endswith('.csv'):
            try: info += f' | {len(pd.read_csv(dst))} baris'
            except: pass
        elif fname.endswith('.npy'):
            try: info += f' | {len(np.load(dst))} indices'
            except: pass
        print(f'  ✔ {fname:<45} {info}')
    else: print(f'  ❌ {fname} — TIDAK DITEMUKAN')
print('Restoring process done!')

# ── Model ─────────────────────────────────────────────────────────────────────
BERT_MODEL  = 'indobenchmark/indobert-base-p2'
MAX_LEN     = 128
NUM_CLASSES = 3
LABEL2ID    = {'positive': 0, 'negative': 1, 'neutral': 2}
ID2LABEL    = {0: 'positive', 1: 'negative', 2: 'neutral'}
LABEL_NAMES = ['positive', 'negative', 'neutral']

# ── Fixed Hyperparameter Tier 2 ───────────────────────────────────────────────
MAX_EPOCHS   = 10
PATIENCE     = 3
N_FOLDS      = 5
WARMUP_RATIO = 0.1
ADAM_EPS     = 1e-8
CLS_DROPOUT  = 0.1   # dropout pada [CLS] path sebelum concatenate
DENSE_SIZE   = 256   # dense layer setelah concatenate

# ── CNN Default (fixed selama Phase 1) ───────────────────────────────────────
# GELU sebagai default — konsisten dengan activation internal IndoBERT
CNN_DEFAULT = {
    'ngram'      : [1, 2, 3],
    'filter_size': 128,
    'dropout'    : 0.3,
    'activation' : 'gelu',
}

# ── Grid Search ───────────────────────────────────────────────────────────────
# lr_bert: range fine-tuning BERT standar (Devlin et al., 2019)
# lr_cnn : low & high bound CNN from scratch (Kim, 2014)
# weight_decay: fix 0.01 standar Hugging Face, non-bias/LayerNorm only
PHASE1_GRID = {
    'lr_bert'     : [1e-5, 2e-5, 3e-5],
    'lr_cnn'      : [1e-4, 1e-3],
    'batch_size'  : [16, 32],
    'weight_decay': [0.01],
}
# Phase 2A: ngram [1,2,3] — unigram, bigram, trigram
PHASE2_GRID_A = {
    'ngram'      : [[1, 2, 3]],
    'filter_size': [128, 256],
    'dropout'    : [0.3, 0.5],
    'activation' : ['relu', 'gelu', 'elu'],
}
# Phase 2B: ngram [2,3,4] — bigram, trigram, 4-gram
PHASE2_GRID_B = {
    'ngram'      : [[2, 3, 4]],
    'filter_size': [128, 256],
    'dropout'    : [0.3, 0.5],
    'activation' : ['relu', 'gelu', 'elu'],
}

# ── Best Config (update manual setelah setiap phase) ──────────────────────────
BEST_LR_BERT    = 1e-5  # contoh: 1e-5
BEST_LR_CNN     = 1e-4  # contoh: 1e-4
BEST_BS         = 32  # contoh: 32
BEST_WD         = 0.01  # fixed
BEST_NGRAM      = [1,2,3]  # contoh: [1,2,3]
BEST_FILTER     = 256  # contoh: 256
BEST_DROPOUT    = 0.5  # contoh: 0.5
BEST_ACTIVATION = 'elu'  # contoh: 'elu'

p1  = len(list(itertools.product(*PHASE1_GRID.values())))
p2a = len(list(itertools.product(*PHASE2_GRID_A.values())))
p2b = len(list(itertools.product(*PHASE2_GRID_B.values())))
print('✅ Konfigurasi berhasil dimuat.')
print(f'   Arsitektur : Dual-Path [CLS]+CNN → Concat → Dense({DENSE_SIZE}) → Dense({NUM_CLASSES})')
print(f'   Phase 1    : {p1} kombinasi × {N_FOLDS} fold = {p1*N_FOLDS} runs (~9–10 jam)')
print(f'   Phase 2A   : {p2a} kombinasi × {N_FOLDS} fold = {p2a*N_FOLDS} runs (~9 jam)')
print(f'   Phase 2B   : {p2b} kombinasi × {N_FOLDS} fold = {p2b*N_FOLDS} runs (~9 jam)')
print(f'   Phase 3    : 2 kondisi × {N_FOLDS} fold = {2*N_FOLDS} runs + evaluasi final')

## 2. Load & Split Data

> **Jalankan di awal SETIAP sesi.**

In [ ]:
print('Memuat data berlabel...')
df = pd.read_csv(INPUT_PATH, low_memory=False)
df = df[df['label'].isin(['positive','negative','neutral'])].copy()
df['label_id'] = df['label'].map(LABEL2ID)
df = df.dropna(subset=['text_bert','label_id']).reset_index(drop=True)
print(f'Total data berlabel: {len(df):,}')
for lbl, cnt in df['label'].value_counts().items():
    print(f'  {lbl:12s}: {cnt:,} ({cnt/len(df)*100:.1f}%)')

test_idx_path = f'{OUTPUT_DIR}/test_set_indices.npy'
if os.path.exists(test_idx_path):
    test_indices = np.load(test_idx_path)
    mask         = np.ones(len(df), dtype=bool)
    mask[test_indices] = False
    df_trainval  = df[mask].reset_index(drop=True)
    df_test      = df.iloc[test_indices].reset_index(drop=True)
    print(f'\nTest set index dimuat dari file existing.')
else:
    tv_idx, test_indices = train_test_split(
        np.arange(len(df)), test_size=0.2,
        random_state=SEED, stratify=df['label_id'].values
    )
    np.save(test_idx_path, test_indices)
    df_trainval = df.iloc[tv_idx].reset_index(drop=True)
    df_test     = df.iloc[test_indices].reset_index(drop=True)
    df_test.to_csv(f'{OUTPUT_DIR}/test_set_fixed.csv', index=False)
    print(f'\nTest set baru dibuat dan index disimpan.')

print(f'Train+Val : {len(df_trainval):,} | Test (FIXED): {len(df_test):,}')
print('Distribusi test set:')
for lbl, cnt in df_test['label'].value_counts().items():
    print(f'  {lbl:12s}: {cnt:,} ({cnt/len(df_test)*100:.1f}%)')

## 3. Dataset Class & Arsitektur Model

> **Jalankan di awal SETIAP sesi.**

Arsitektur **Dual-Path** memiliki dua path paralel:
- **Path 1 ([CLS])**: token pertama `last_hidden_state[:,0,:]` — representasi konteks global
- **Path 2 (CNN)**: Conv1D multi-kernel pada sequence — representasi pola lokal

Keduanya di-concatenate sebelum classifier. Input classifier: `768 + filter_size × n_kernels`.

In [ ]:
print(f'Memuat tokenizer: {BERT_MODEL}...')
tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)
print('✅ Tokenizer berhasil dimuat.')


class MBGDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts, self.labels      = texts, labels
        self.tokenizer, self.max_len = tokenizer, max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]), max_length=self.max_len,
            padding='max_length', truncation=True, return_tensors='pt'
        )
        return {
            'input_ids'     : enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label'         : torch.tensor(self.labels[idx], dtype=torch.long)
        }


class IndoBERTCNN(nn.Module):
    """
    Proposed Model: Dual-Path IndoBERT + CNN 1D.

    Arsitektur:
        BERT → last_hidden_state
                  |
      ┌───────────┴───────────┐
      Path 1: [CLS]     Path 2: CNN multi-kernel
      konteks global    pola n-gram lokal
      Dropout(cls_drop) GlobalMaxPool
      └───── Concatenate ─────┘
             Dropout → Dense(dense_size) → Dropout → Dense(num_classes)

    CNN melengkapi [CLS] — tidak menggantikan.
    Ref: Chen et al. (2020); Devlin et al. (2019); Kim (2014).
    """
    def __init__(self, bert_model_name, num_classes,
                 ngram_sizes, filter_size, dropout,
                 activation, cls_dropout=0.1, dense_size=256):
        super().__init__()
        self.bert    = AutoModel.from_pretrained(bert_model_name)
        hidden       = self.bert.config.hidden_size  # 768

        def get_act(name):
            if name == 'gelu': return nn.GELU()
            if name == 'elu':  return nn.ELU(alpha=1.0)
            return nn.ReLU()

        # Path 2: CNN multi-kernel
        self.convs    = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(hidden, filter_size, kernel_size=k, padding=k//2),
                get_act(activation)
            ) for k in ngram_sizes
        ])
        # Path 1: [CLS] dropout
        self.cls_drop = nn.Dropout(cls_dropout)
        # Classifier head setelah concatenate
        combined_dim  = hidden + filter_size * len(ngram_sizes)
        self.drop1    = nn.Dropout(dropout)
        self.fc1      = nn.Linear(combined_dim, dense_size)
        self.act_fc   = get_act(activation)
        self.drop2    = nn.Dropout(dropout)
        self.fc2      = nn.Linear(dense_size, num_classes)

    def forward(self, input_ids, attention_mask):
        bert_out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        seq      = bert_out.last_hidden_state        # [batch, 128, 768]
        # ── Path 1: [CLS] — konteks global ──────────────────────────────────
        cls_repr = self.cls_drop(seq[:, 0, :])       # [batch, 768]
        # ── Path 2: CNN — pola n-gram lokal ─────────────────────────────────
        x        = seq.permute(0, 2, 1)              # [batch, 768, 128]
        pooled   = [F.adaptive_max_pool1d(conv(x), 1).squeeze(-1) for conv in self.convs]
        cnn_repr = torch.cat(pooled, dim=1)          # [batch, filter×n_kernels]
        # ── Concatenate + Classifier ─────────────────────────────────────────
        combined = torch.cat([cls_repr, cnn_repr], dim=1)
        x        = self.drop1(combined)
        x        = self.act_fc(self.fc1(x))
        return self.fc2(self.drop2(x))


# Validasi arsitektur
print('Validasi arsitektur Dual-Path...')
_dm = IndoBERTCNN(BERT_MODEL, NUM_CLASSES, CNN_DEFAULT['ngram'],
                  CNN_DEFAULT['filter_size'], CNN_DEFAULT['dropout'],
                  CNN_DEFAULT['activation'], CLS_DROPOUT, DENSE_SIZE).to(DEVICE)
_di = torch.zeros(2, MAX_LEN, dtype=torch.long).to(DEVICE)
_dk = torch.ones(2, MAX_LEN, dtype=torch.long).to(DEVICE)
with torch.no_grad(): _do = _dm(_di, _dk)
_h   = _dm.bert.config.hidden_size
_cnn = CNN_DEFAULT['filter_size'] * len(CNN_DEFAULT['ngram'])
print(f'  ✅ Output shape  : {list(_do.shape)} (expected [2, {NUM_CLASSES}])')
print(f'  ✅ [CLS] dim     : {_h}')
print(f'  ✅ CNN dim       : {_cnn} ({CNN_DEFAULT["filter_size"]}×{len(CNN_DEFAULT["ngram"])} kernels)')
print(f'  ✅ Combined dim  : {_h+_cnn} → Dense({DENSE_SIZE}) → Dense({NUM_CLASSES})')
del _dm, _di, _dk, _do; torch.cuda.empty_cache()
print('\n✅ Arsitektur Dual-Path [CLS]+CNN berhasil didefinisikan.')

## 4. Training Utilities

> **Jalankan di awal SETIAP sesi.**

In [ ]:
def compute_metrics(y_true, y_pred):
    return {
        'accuracy'   : round(accuracy_score(y_true, y_pred), 4),
        'precision'  : round(precision_score(y_true, y_pred, average='macro', zero_division=0), 4),
        'recall'     : round(recall_score(y_true, y_pred, average='macro', zero_division=0), 4),
        'f1_macro'   : round(f1_score(y_true, y_pred, average='macro', zero_division=0), 4),
        'f1_weighted': round(f1_score(y_true, y_pred, average='weighted', zero_division=0), 4),
    }


def train_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss, preds_all, labels_all = 0.0, [], []
    for batch in loader:
        ids, mask, lbls = (batch['input_ids'].to(device),
                           batch['attention_mask'].to(device),
                           batch['label'].to(device))
        optimizer.zero_grad()
        logits = model(ids, mask)
        loss   = criterion(logits, lbls)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step(); scheduler.step()
        total_loss += loss.item()
        preds_all.extend(torch.argmax(logits, 1).cpu().numpy())
        labels_all.extend(lbls.cpu().numpy())
    return total_loss/len(loader), f1_score(labels_all, preds_all, average='macro', zero_division=0)


def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, preds_all, labels_all, probs_all = 0.0, [], [], []
    with torch.no_grad():
        for batch in loader:
            ids, mask, lbls = (batch['input_ids'].to(device),
                               batch['attention_mask'].to(device),
                               batch['label'].to(device))
            logits = model(ids, mask)
            total_loss += criterion(logits, lbls).item()
            preds_all.extend(torch.argmax(logits, 1).cpu().numpy())
            labels_all.extend(lbls.cpu().numpy())
            probs_all.append(torch.softmax(logits, dim=1).cpu().numpy())
    return (total_loss/len(loader), compute_metrics(labels_all, preds_all),
            preds_all, labels_all, np.vstack(probs_all))


def apply_undersampling(X, y, seed=SEED):
    min_n = min(np.bincount(y.astype(int)))
    X_res, y_res = [], []
    for cls in np.unique(y):
        idx = np.where(y == cls)[0]
        s   = resample(idx, n_samples=min_n, random_state=seed, replace=False)
        X_res.append(X[s]); y_res.append(y[s])
    X_res, y_res = np.concatenate(X_res), np.concatenate(y_res)
    return X_res[np.random.RandomState(seed).permutation(len(X_res))], \
           y_res[np.random.RandomState(seed).permutation(len(y_res))]


def build_criterion(y_train, condition, device):
    """Class weight manual — shape selalu [NUM_CLASSES]."""
    if condition == 'S2': return nn.CrossEntropyLoss()
    n      = len(y_train)
    w      = np.zeros(NUM_CLASSES, dtype=np.float32)
    for c in range(NUM_CLASSES):
        nc = np.sum(y_train == c)
        w[c] = (n / (NUM_CLASSES * nc)) if nc > 0 else 0.0
    return nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float).to(device))


def build_optimizer(model, lr_bert, lr_cnn, weight_decay):
    """
    4 parameter groups — Differential LR + Weight Decay Exclusion:
      G1: BERT weight  → lr_bert, weight_decay
      G2: BERT bias/LN → lr_bert, wd=0.0
      G3: CNN+head wt  → lr_cnn,  weight_decay
      G4: CNN+head bias→ lr_cnn,  wd=0.0
    Ref: Howard & Ruder (2018); Loshchilov & Hutter (2019).
    """
    no_decay     = ['bias', 'LayerNorm.weight', 'LayerNorm.bias']
    bert_decay   = [p for n,p in model.bert.named_parameters()
                    if not any(nd in n for nd in no_decay) and p.requires_grad]
    bert_nodecay = [p for n,p in model.bert.named_parameters()
                    if any(nd in n for nd in no_decay) and p.requires_grad]
    cnn_decay    = [p for n,p in model.named_parameters()
                    if not n.startswith('bert.') and
                    not any(nd in n for nd in no_decay) and p.requires_grad]
    cnn_nodecay  = [p for n,p in model.named_parameters()
                    if not n.startswith('bert.') and
                    any(nd in n for nd in no_decay) and p.requires_grad]
    pg = [
        {'params': bert_decay,   'lr': lr_bert, 'weight_decay': weight_decay},
        {'params': bert_nodecay, 'lr': lr_bert, 'weight_decay': 0.0},
        {'params': cnn_decay,    'lr': lr_cnn,  'weight_decay': weight_decay},
        {'params': cnn_nodecay,  'lr': lr_cnn,  'weight_decay': 0.0},
    ]
    n_t = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_g = sum(p.numel() for g in pg for p in g['params'])
    assert n_t == n_g, f'Param mismatch: {n_t} vs {n_g}'
    return AdamW(pg, eps=ADAM_EPS)


def run_kfold(df_tv, lr_bert, lr_cnn, batch_size, weight_decay,
              ngram, filter_size, dropout, activation,
              data_condition='S1', combo_id=None, combo_label=''):
    """
    Stratified K-Fold dengan full recording:
    - epoch_history : loss + metrik per epoch, per fold, SEMUA kombinasi
    - fold_details  : metrik terbaik per fold
    """
    skf           = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    X_tv, y_tv    = df_tv['text_bert'].values, df_tv['label_id'].values
    fold_metrics  = []; epoch_history = []; fold_details = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_tv, y_tv), 1):
        X_tr, y_tr   = X_tv[tr_idx], y_tv[tr_idx]
        X_val, y_val = X_tv[val_idx], y_tv[val_idx]
        if data_condition == 'S2': X_tr, y_tr = apply_undersampling(X_tr, y_tr)
        criterion = build_criterion(y_tr, data_condition, DEVICE)
        # num_workers: 2 Colab | 0 Kaggle (ubah ke 0 jika Kaggle)
        train_dl = DataLoader(MBGDataset(X_tr, y_tr, tokenizer, MAX_LEN),
                              batch_size=batch_size, shuffle=True,  num_workers=0)
        val_dl   = DataLoader(MBGDataset(X_val, y_val, tokenizer, MAX_LEN),
                              batch_size=batch_size, shuffle=False, num_workers=0)
        model = IndoBERTCNN(BERT_MODEL, NUM_CLASSES, ngram, filter_size,
                            dropout, activation, CLS_DROPOUT, DENSE_SIZE).to(DEVICE)
        optimizer   = build_optimizer(model, lr_bert, lr_cnn, weight_decay)
        total_steps = len(train_dl) * MAX_EPOCHS
        scheduler   = get_linear_schedule_with_warmup(
            optimizer, int(total_steps * WARMUP_RATIO), total_steps)
        best_val_loss = float('inf'); patience_cnt = 0
        best_metrics  = None;        best_epoch   = 0
        epoch_bar = tqdm(range(1, MAX_EPOCHS+1), desc=f'  Fold {fold}/{N_FOLDS}',
                         leave=False, bar_format='{l_bar}{bar:20}{r_bar}')
        for epoch in epoch_bar:
            tr_loss, _           = train_epoch(model, train_dl, optimizer, scheduler, criterion, DEVICE)
            val_loss, m, _, _, _ = eval_epoch(model, val_dl, criterion, DEVICE)
            epoch_history.append({
                'combo_id':combo_id,'combo_label':combo_label,
                'condition':data_condition,'fold':fold,'epoch':epoch,
                'train_loss':round(tr_loss,4),'val_loss':round(val_loss,4),
                **{f'val_{k}':v for k,v in m.items()},'is_best_epoch':False
            })
            epoch_bar.set_postfix({'tr':f'{tr_loss:.4f}','val':f'{val_loss:.4f}',
                                   'f1':f'{m["f1_macro"]:.4f}','pat':f'{patience_cnt}/{PATIENCE}'})
            if val_loss < best_val_loss:
                best_val_loss = val_loss; best_metrics = m
                patience_cnt  = 0;       best_epoch   = epoch
            else:
                patience_cnt += 1
                if patience_cnt >= PATIENCE:
                    epoch_bar.set_description(f'  Fold {fold}/{N_FOLDS} [stop ep{epoch}]'); break
        for rec in epoch_history:
            if rec['combo_id']==combo_id and rec['fold']==fold and rec['epoch']==best_epoch:
                rec['is_best_epoch'] = True
        fold_details.append({
            'combo_id':combo_id,'combo_label':combo_label,'condition':data_condition,
            'fold':fold,'best_epoch':best_epoch,'best_val_loss':round(best_val_loss,4),
            **{f'val_{k}':v for k,v in best_metrics.items()}
        })
        fold_metrics.append(best_metrics)
        print(f'  ├─ Fold {fold}/{N_FOLDS} → F1={best_metrics["f1_macro"]:.4f} | '
              f'Acc={best_metrics["accuracy"]:.4f} | best_ep={best_epoch}')
        del model; torch.cuda.empty_cache()

    keys = fold_metrics[0].keys()
    avg  = {k: round(np.mean([m[k] for m in fold_metrics]),4) for k in keys}
    std  = {f'{k}_std': round(np.std([m[k] for m in fold_metrics]),4) for k in keys}
    return {**avg, **std, 'epoch_history':epoch_history, 'fold_details':fold_details}


# ── Visualisasi helpers ───────────────────────────────────────────────────────
def plot_learning_curves(epoch_history, combo_id, combo_label, save_path):
    df_h = pd.DataFrame([r for r in epoch_history if r['combo_id']==combo_id])
    if df_h.empty: return
    fig, axes = plt.subplots(1,2,figsize=(14,5))
    colors = ['#534AB7','#E24B4A','#1D9E75','#EF9F27','#AFA9EC']
    for fi,(fn,grp) in enumerate(df_h.groupby('fold')):
        c  = colors[fi%len(colors)]
        be = grp[grp['is_best_epoch']]['epoch'].values
        axes[0].plot(grp['epoch'],grp['train_loss'],'--',color=c,alpha=0.5,linewidth=1)
        axes[0].plot(grp['epoch'],grp['val_loss'],'-',color=c,alpha=0.9,linewidth=1.5,label=f'Fold {fn}')
        if len(be): axes[0].scatter(be[0],grp[grp['epoch']==be[0]]['val_loss'].values[0],color=c,s=60,zorder=5)
        axes[1].plot(grp['epoch'],grp['val_f1_macro'],'-',color=c,linewidth=1.5,label=f'Fold {fn}')
    for ax,t,y in [(axes[0],f'Loss Curves\n{combo_label}','Loss'),(axes[1],f'Val F1-Macro\n{combo_label}','F1-Macro')]:
        ax.set_title(t,fontsize=11); ax.set_xlabel('Epoch'); ax.set_ylabel(y)
        ax.legend(fontsize=8); sns.despine(ax=ax)
    plt.tight_layout(); plt.savefig(save_path,dpi=150,bbox_inches='tight'); plt.close()
    print(f'     ✔ Learning curve: {save_path.split("/")[-1]}')

def plot_kfold_stability(fold_details_all, phase_label, save_path):
    df_fd = pd.DataFrame(fold_details_all)
    if df_fd.empty: return
    n = df_fd['combo_label'].nunique()
    fig,ax = plt.subplots(figsize=(max(10,n*0.8),5))
    df_fd.pivot_table(index='fold',columns='combo_label',values='val_f1_macro').boxplot(ax=ax,grid=False)
    ax.set_title(f'Stabilitas K-Fold — {phase_label}',fontsize=12)
    ax.set_ylabel('F1-Macro per Fold'); ax.set_xlabel('Konfigurasi')
    ax.tick_params(axis='x',rotation=45,labelsize=7)
    sns.despine(ax=ax); plt.tight_layout()
    plt.savefig(save_path,dpi=150,bbox_inches='tight'); plt.close()
    print(f'  ✔ Stabilitas K-Fold: {save_path.split("/")[-1]}')

def plot_confusion_matrix_pair(y_true, y_pred, title_suffix, save_prefix):
    cm      = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype(float)/cm.sum(axis=1,keepdims=True)
    fig,axes = plt.subplots(1,2,figsize=(14,6))
    for ax,data,fmt,title in [
        (axes[0],cm,'d',f'Confusion Matrix (count)\n{title_suffix}'),
        (axes[1],cm_norm,'.2%',f'Confusion Matrix (normalized %)\n{title_suffix}')
    ]:
        sns.heatmap(data,annot=True,fmt=fmt,cmap='Blues',
                    xticklabels=LABEL_NAMES,yticklabels=LABEL_NAMES,ax=ax)
        ax.set_title(title,fontsize=11); ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    plt.tight_layout(); plt.savefig(f'{save_prefix}_cm_pair.png',dpi=150,bbox_inches='tight'); plt.close()
    print('  ✔ Confusion matrix pair disimpan.')


print('✅ Training utilities berhasil didefinisikan.')
print('   Optimizer  : AdamW differential LR + weight decay exclusion (4 param groups)')
print('   Arsitektur : Dual-Path [CLS]+CNN → Concatenate → Dense → Classifier')
print('   Recording  : epoch_history (semua kombinasi) + fold_details')

---
# PHASE 1 — Differential LR Search

**Grid**: 3×2×2×1 = **12 kombinasi × 5 fold = 60 runs (~9–10 jam)**

| Parameter | Kandidat | Justifikasi |
|---|---|---|
| `lr_bert` | 1e-5, 2e-5, 3e-5 | Fine-tuning BERT (Devlin et al., 2019) |
| `lr_cnn` | 1e-4, 1e-3 | Training from scratch CNN (Kim, 2014) |
| `batch_size` | 16, 32 | — |
| `weight_decay` | 0.01 (fixed) | Non-bias/LayerNorm only (Loshchilov & Hutter, 2019) |

CNN fixed: ngram=[1,2,3], filter=128, dropout=0.3, activation=gelu

In [ ]:
p1_combos = list(itertools.product(
    PHASE1_GRID['lr_bert'], PHASE1_GRID['lr_cnn'], PHASE1_GRID['batch_size']))
print(f'Phase 1: {len(p1_combos)} kombinasi × {N_FOLDS} fold = {len(p1_combos)*N_FOLDS} runs')
print(f'CNN default: {CNN_DEFAULT}')
for i,(lb,lc,bs) in enumerate(p1_combos,1):
    print(f'  [{i:02d}] lr_bert={lb:.0e} | lr_cnn={lc:.0e} | bs={bs}')

In [ ]:
p1_results=[]; p1_epoch_history=[]; p1_fold_details=[]
t_start_total = time.time()
combo_bar = tqdm(enumerate(p1_combos,1),total=len(p1_combos),
                 desc='Phase 1',bar_format='{l_bar}{bar:25}{r_bar}')
for i,(lb,lc,bs) in combo_bar:
    combo_label = f'lb={lb:.0e}_lc={lc:.0e}_bs={bs}'
    combo_bar.set_description(f'Phase 1 [{i:02d}/{len(p1_combos)}]')
    combo_bar.set_postfix({'lr_bert':f'{lb:.0e}','lr_cnn':f'{lc:.0e}','bs':bs})
    print(f'\n{"─"*65}\n[{i:02d}/{len(p1_combos)}] {combo_label}\n{"─"*65}')
    t_start = time.time()
    res = run_kfold(df_trainval,lr_bert=lb,lr_cnn=lc,batch_size=bs,weight_decay=BEST_WD,
                    ngram=CNN_DEFAULT['ngram'],filter_size=CNN_DEFAULT['filter_size'],
                    dropout=CNN_DEFAULT['dropout'],activation=CNN_DEFAULT['activation'],
                    data_condition='S1',combo_id=i,combo_label=combo_label)
    elapsed=time.time()-t_start
    p1_epoch_history.extend(res.pop('epoch_history'))
    p1_fold_details.extend(res.pop('fold_details'))
    avg_time=(time.time()-t_start_total)/i; remaining=avg_time*(len(p1_combos)-i)
    rem_h,rem_m=int(remaining//3600),int((remaining%3600)//60)
    p1_results.append({'combo_id':i,'lr_bert':lb,'lr_cnn':lc,'batch_size':bs,
                       'weight_decay':BEST_WD,'combo_label':combo_label,**res})
    print(f'  └─ SELESAI | F1-Macro: {res["f1_macro"]:.4f} ± {res["f1_macro_std"]:.4f}')
    print(f'     Acc={res["accuracy"]:.4f} | Prec={res["precision"]:.4f} | Rec={res["recall"]:.4f}')
    print(f'     Waktu: {elapsed/60:.1f} menit | Sisa: {rem_h}j {rem_m}m ({len(p1_combos)-i} tersisa)')
    pd.DataFrame(p1_results).to_csv(f'{OUTPUT_DIR}/phase1_results.csv',index=False)
    pd.DataFrame(p1_epoch_history).to_csv(f'{OUTPUT_DIR}/phase1_epoch_history.csv',index=False)
    pd.DataFrame(p1_fold_details).to_csv(f'{OUTPUT_DIR}/phase1_fold_details.csv',index=False)
    print('     ✔ Progress disimpan')

total_time=time.time()-t_start_total
df_p1=pd.DataFrame(p1_results).sort_values('f1_macro',ascending=False)
best_p1=df_p1.iloc[0]
top_cols=['combo_id','lr_bert','lr_cnn','batch_size','weight_decay',
          'f1_macro','f1_macro_std','f1_weighted','accuracy','precision','recall']
df_p1.head(10)[top_cols].to_csv(f'{OUTPUT_DIR}/phase1_summary_top10.csv',index=False)
print(f'\n{"="*65}\n PHASE 1 SELESAI — {total_time/3600:.1f} jam\n{"="*65}')
print(df_p1[top_cols].to_string(index=False))
print(f'\n Config Terbaik: lr_bert={best_p1["lr_bert"]} | lr_cnn={best_p1["lr_cnn"]} | bs={int(best_p1["batch_size"])}')
print(f'   F1-Macro: {best_p1["f1_macro"]:.4f} ± {best_p1["f1_macro_std"]:.4f}')
print('\n⚠️  Update BEST_LR_BERT, BEST_LR_CNN, BEST_BS di sel Konfigurasi sebelum Phase 2A!')

In [ ]:
df_p1_v=pd.DataFrame(p1_results)
hm=df_p1_v.groupby(['lr_bert','lr_cnn'])['f1_macro'].mean().unstack()
fig,axes=plt.subplots(1,2,figsize=(16,5))
sns.heatmap(hm,annot=True,fmt='.4f',cmap='YlOrRd',ax=axes[0],
            xticklabels=[f'{v:.0e}' for v in hm.columns],
            yticklabels=[f'{v:.0e}' for v in hm.index])
axes[0].set_xlabel('lr_cnn',fontsize=11); axes[0].set_ylabel('lr_bert',fontsize=11)
axes[0].set_title('Heatmap F1-Macro: lr_bert × lr_cnn\nDual-Path [CLS]+CNN',fontsize=11)
df_top=df_p1_v.sort_values('f1_macro',ascending=False).head(10).reset_index(drop=True)
df_top['cfg']=df_top.apply(lambda r:f"lb={r['lr_bert']:.0e}\nlc={r['lr_cnn']:.0e}\nbs={int(r['batch_size'])}",axis=1)
cs=['#1D9E75']+['#534AB7']*9
axes[1].bar(range(len(df_top)),df_top['f1_macro'],color=cs,edgecolor='none',alpha=0.9)
axes[1].errorbar(range(len(df_top)),df_top['f1_macro'],yerr=df_top['f1_macro_std'],fmt='none',ecolor='#888780',capsize=4)
axes[1].set_xticks(range(len(df_top))); axes[1].set_xticklabels(df_top['cfg'],fontsize=7)
axes[1].set_ylabel('F1-Macro (mean 5-fold)'); axes[1].set_title('Top-10 Phase 1 (hijau=terbaik)')
axes[1].set_ylim(max(0,df_top['f1_macro'].min()-0.03),min(1,df_top['f1_macro'].max()+0.03))
sns.despine(ax=axes[1])
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/phase1_results.png',dpi=150,bbox_inches='tight')
plt.close()
print('  ✔ phase1_results.png tersimpan')

fig,axes=plt.subplots(1,2,figsize=(14,5))
clrs=['#534AB7','#E24B4A','#1D9E75']
for idx,lc in enumerate(PHASE1_GRID['lr_cnn']):
    sub=df_p1_v[df_p1_v['lr_cnn']==lc].groupby('lr_bert')['f1_macro'].mean()
    axes[0].plot([f'{v:.0e}' for v in sub.index],sub.values,marker='o',color=clrs[idx],label=f'lr_cnn={lc:.0e}',linewidth=1.8)
axes[0].set_xlabel('lr_bert'); axes[0].set_ylabel('F1-Macro')
axes[0].set_title('Sensitivitas lr_bert per lr_cnn'); axes[0].legend(fontsize=9); sns.despine(ax=axes[0])
for idx,lb in enumerate(PHASE1_GRID['lr_bert']):
    sub=df_p1_v[df_p1_v['lr_bert']==lb].groupby('lr_cnn')['f1_macro'].mean()
    axes[1].plot([f'{v:.0e}' for v in sub.index],sub.values,marker='o',color=clrs[idx],label=f'lr_bert={lb:.0e}',linewidth=1.8)
axes[1].set_xlabel('lr_cnn'); axes[1].set_ylabel('F1-Macro')
axes[1].set_title('Sensitivitas lr_cnn per lr_bert'); axes[1].legend(fontsize=9); sns.despine(ax=axes[1])
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/phase1_lr_sensitivity.png', dpi=150, bbox_inches='tight')
plt.close()
print('  ✔ phase1_lr_sensitivity.png tersimpan')

plot_learning_curves(p1_epoch_history,int(best_p1['combo_id']),best_p1['combo_label'],
                     f'{OUTPUT_DIR}/phase1_learning_curves_best.png')
plot_kfold_stability(p1_fold_details,'Phase 1',f'{OUTPUT_DIR}/phase1_kfold_stability.png')
print('✔ Semua visualisasi Phase 1 tersimpan.')

---
# PHASE 2A — CNN Search: Ngram [1,2,3]

**Prasyarat**: `BEST_LR_BERT`, `BEST_LR_CNN`, `BEST_BS` sudah diupdate.  
**Grid**: 1×2×2×3 = **12 kombinasi × 5 fold = 60 runs (~9 jam)**

> ⏱️ Estimasi: **~9 jam** GPU T4.

In [ ]:
assert BEST_LR_BERT is not None,'❌ BEST_LR_BERT belum diisi!'
assert BEST_LR_CNN  is not None,'❌ BEST_LR_CNN belum diisi!'
assert BEST_BS      is not None,'❌ BEST_BS belum diisi!'
p2a_combos=list(itertools.product(
    PHASE2_GRID_A['ngram'],PHASE2_GRID_A['filter_size'],
    PHASE2_GRID_A['dropout'],PHASE2_GRID_A['activation']))
print(f'Phase 2A: {len(p2a_combos)} kombinasi × {N_FOLDS} fold = {len(p2a_combos)*N_FOLDS} runs')
print(f'BERT fixed: lr_bert={BEST_LR_BERT} | lr_cnn={BEST_LR_CNN} | bs={BEST_BS} | wd={BEST_WD}')
for i,(ng,fs,dr,ac) in enumerate(p2a_combos,1):
    print(f'  [{i:02d}] ngram={ng} | filter={fs} | dropout={dr} | act={ac}')

In [ ]:
p2a_results=[]; p2a_epoch_history=[]; p2a_fold_details=[]
t_start_total=time.time()
combo_bar=tqdm(enumerate(p2a_combos,1),total=len(p2a_combos),
               desc='Phase 2A',bar_format='{l_bar}{bar:25}{r_bar}')
for i,(ng,fs,dr,ac) in combo_bar:
    combo_label=f'ng={ng}_f={fs}_dr={dr}_{ac}'
    combo_bar.set_description(f'Phase 2A [{i:02d}/{len(p2a_combos)}]')
    combo_bar.set_postfix({'f':fs,'dr':dr,'act':ac})
    print(f'\n{"─"*65}\n[{i:02d}/{len(p2a_combos)}] {combo_label}\n{"─"*65}')
    t_start=time.time()
    res=run_kfold(df_trainval,lr_bert=BEST_LR_BERT,lr_cnn=BEST_LR_CNN,
                  batch_size=BEST_BS,weight_decay=BEST_WD,
                  ngram=ng,filter_size=fs,dropout=dr,activation=ac,
                  data_condition='S1',combo_id=i,combo_label=combo_label)
    elapsed=time.time()-t_start
    p2a_epoch_history.extend(res.pop('epoch_history'))
    p2a_fold_details.extend(res.pop('fold_details'))
    avg_time=(time.time()-t_start_total)/i; remaining=avg_time*(len(p2a_combos)-i)
    rem_h,rem_m=int(remaining//3600),int((remaining%3600)//60)
    p2a_results.append({'combo_id':i,'batch':'A','ngram':str(ng),'filter_size':fs,
                        'dropout':dr,'activation':ac,'combo_label':combo_label,**res})
    print(f'  └─ SELESAI | F1-Macro: {res["f1_macro"]:.4f} ± {res["f1_macro_std"]:.4f}')
    print(f'     Acc={res["accuracy"]:.4f} | Prec={res["precision"]:.4f} | Rec={res["recall"]:.4f}')
    print(f'     Waktu: {elapsed/60:.1f} menit | Sisa: {rem_h}j {rem_m}m ({len(p2a_combos)-i} tersisa)')
    pd.DataFrame(p2a_results).to_csv(f'{OUTPUT_DIR}/phase2A_results.csv',index=False)
    pd.DataFrame(p2a_epoch_history).to_csv(f'{OUTPUT_DIR}/phase2A_epoch_history.csv',index=False)
    pd.DataFrame(p2a_fold_details).to_csv(f'{OUTPUT_DIR}/phase2A_fold_details.csv',index=False)
    print('     ✔ Progress disimpan')

total_time=time.time()-t_start_total
df_p2a=pd.DataFrame(p2a_results).sort_values('f1_macro',ascending=False)
best_p2a=df_p2a.iloc[0]
print(f'\n{"="*65}\n PHASE 2A SELESAI — {total_time/3600:.1f} jam\n{"="*65}')
print(df_p2a[['combo_id','ngram','filter_size','dropout','activation',
               'f1_macro','f1_macro_std','accuracy']].to_string(index=False))
print(f'\n Terbaik 2A: filter={int(best_p2a["filter_size"])} | dropout={best_p2a["dropout"]} | act={best_p2a["activation"]}')
print('\n Lanjut ke Phase 2B di sesi berikutnya.')

---
# PHASE 2B — CNN Search: Ngram [2,3,4]

**Prasyarat**: Phase 2A sudah selesai.  
**Grid**: 1×2×2×3 = **12 kombinasi × 5 fold = 60 runs (~9 jam)**

> ⏱️ Estimasi: **~9 jam** GPU T4.

In [ ]:
assert BEST_LR_BERT is not None,'❌ BEST_LR_BERT belum diisi!'
assert BEST_LR_CNN  is not None,'❌ BEST_LR_CNN belum diisi!'
assert BEST_BS      is not None,'❌ BEST_BS belum diisi!'
assert os.path.exists(f'{OUTPUT_DIR}/phase2A_results.csv'), \
    '❌ phase2A_results.csv tidak ditemukan! Jalankan Phase 2A terlebih dahulu.'
p2b_combos=list(itertools.product(
    PHASE2_GRID_B['ngram'],PHASE2_GRID_B['filter_size'],
    PHASE2_GRID_B['dropout'],PHASE2_GRID_B['activation']))
print(f'Phase 2B: {len(p2b_combos)} kombinasi × {N_FOLDS} fold = {len(p2b_combos)*N_FOLDS} runs')
print(f'BERT fixed: lr_bert={BEST_LR_BERT} | lr_cnn={BEST_LR_CNN} | bs={BEST_BS} | wd={BEST_WD}')
for i,(ng,fs,dr,ac) in enumerate(p2b_combos,1):
    print(f'  [{i:02d}] ngram={ng} | filter={fs} | dropout={dr} | act={ac}')

In [ ]:
p2b_results=[]; p2b_epoch_history=[]; p2b_fold_details=[]
t_start_total=time.time()
combo_bar=tqdm(enumerate(p2b_combos,1),total=len(p2b_combos),
               desc='Phase 2B',bar_format='{l_bar}{bar:25}{r_bar}')
for i,(ng,fs,dr,ac) in combo_bar:
    combo_label=f'ng={ng}_f={fs}_dr={dr}_{ac}'
    combo_bar.set_description(f'Phase 2B [{i:02d}/{len(p2b_combos)}]')
    combo_bar.set_postfix({'f':fs,'dr':dr,'act':ac})
    print(f'\n{"─"*65}\n[{i:02d}/{len(p2b_combos)}] {combo_label}\n{"─"*65}')
    t_start=time.time()
    res=run_kfold(df_trainval,lr_bert=BEST_LR_BERT,lr_cnn=BEST_LR_CNN,
                  batch_size=BEST_BS,weight_decay=BEST_WD,
                  ngram=ng,filter_size=fs,dropout=dr,activation=ac,
                  data_condition='S1',combo_id=i+100,combo_label=combo_label)
    elapsed=time.time()-t_start
    p2b_epoch_history.extend(res.pop('epoch_history'))
    p2b_fold_details.extend(res.pop('fold_details'))
    avg_time=(time.time()-t_start_total)/i; remaining=avg_time*(len(p2b_combos)-i)
    rem_h,rem_m=int(remaining//3600),int((remaining%3600)//60)
    p2b_results.append({'combo_id':i+100,'batch':'B','ngram':str(ng),'filter_size':fs,
                        'dropout':dr,'activation':ac,'combo_label':combo_label,**res})
    print(f'  └─ SELESAI | F1-Macro: {res["f1_macro"]:.4f} ± {res["f1_macro_std"]:.4f}')
    print(f'     Acc={res["accuracy"]:.4f} | Prec={res["precision"]:.4f} | Rec={res["recall"]:.4f}')
    print(f'     Waktu: {elapsed/60:.1f} menit | Sisa: {rem_h}j {rem_m}m ({len(p2b_combos)-i} tersisa)')
    pd.DataFrame(p2b_results).to_csv(f'{OUTPUT_DIR}/phase2B_results.csv',index=False)
    pd.DataFrame(p2b_epoch_history).to_csv(f'{OUTPUT_DIR}/phase2B_epoch_history.csv',index=False)
    pd.DataFrame(p2b_fold_details).to_csv(f'{OUTPUT_DIR}/phase2B_fold_details.csv',index=False)
    print('     ✔ Progress disimpan')

# Gabungkan 2A + 2B
df_p2a_l  = pd.read_csv(f'{OUTPUT_DIR}/phase2A_results.csv')
df_p2_all = pd.concat([df_p2a_l,pd.DataFrame(p2b_results)],ignore_index=True).sort_values('f1_macro',ascending=False)
df_p2_all.to_csv(f'{OUTPUT_DIR}/phase2_combined_results.csv',index=False)
top2=['combo_id','batch','ngram','filter_size','dropout','activation',
      'f1_macro','f1_macro_std','f1_weighted','accuracy','precision','recall']
df_p2_all.head(10)[top2].to_csv(f'{OUTPUT_DIR}/phase2_summary_top10.csv',index=False)
best_p2=df_p2_all.iloc[0]
total_time=time.time()-t_start_total
print(f'\n{"="*65}\n PHASE 2B SELESAI — {total_time/3600:.1f} jam\n{"="*65}')
print('HASIL GABUNGAN Phase 2A + 2B (top-10):')
print(df_p2_all.head(10)[top2].to_string(index=False))
print(f'\n CNN Config Terbaik:')
print(f'   batch={best_p2["batch"]} | ngram={best_p2["ngram"]} | filter={int(best_p2["filter_size"])}')
print(f'   dropout={best_p2["dropout"]} | activation={best_p2["activation"]}')
print(f'   F1-Macro: {best_p2["f1_macro"]:.4f} ± {best_p2["f1_macro_std"]:.4f}')
print('\n⚠️  Update BEST_NGRAM, BEST_FILTER, BEST_DROPOUT, BEST_ACTIVATION sebelum Phase 3!')

In [ ]:
# ── Fallback: load df_p2_all dari CSV jika lintas sesi ───────────────────────
if 'df_p2_all' not in dir():
    _p2_combined = f'{OUTPUT_DIR}/phase2_combined_results.csv'
    if os.path.exists(_p2_combined):
        df_p2_all = pd.read_csv(_p2_combined)
        print('✅ df_p2_all dimuat dari CSV')
    else:
        raise FileNotFoundError(
            '❌ phase2_combined_results.csv tidak ditemukan!\n'
            'Pastikan Phase 2B sudah selesai dan file ada di OUTPUT_DIR.'
        )

df_p2_v=df_p2_all.sort_values('f1_macro',ascending=False).reset_index(drop=True)
df_p2_v['cfg']=df_p2_v.apply(lambda r:f"ng={r['ngram']}\nf={int(r['filter_size'])}\ndr={r['dropout']}\n{r['activation']}",axis=1)
bc=df_p2_v['batch'].map({'A':'#534AB7','B':'#E24B4A'}).tolist(); bc[0]='#1D9E75'
fig,ax=plt.subplots(figsize=(18,5))
ax.bar(range(len(df_p2_v)),df_p2_v['f1_macro'],color=bc,edgecolor='none',alpha=0.9)
ax.errorbar(range(len(df_p2_v)),df_p2_v['f1_macro'],yerr=df_p2_v['f1_macro_std'],fmt='none',ecolor='#888780',capsize=4)
ax.set_xticks(range(len(df_p2_v))); ax.set_xticklabels(df_p2_v['cfg'],fontsize=7)
ax.set_ylabel('F1-Macro (mean 5-fold)')
ax.set_title('Phase 2 A+B — CNN Search (ungu=[1,2,3], merah=[2,3,4], hijau=terbaik)')
ax.legend(handles=[mpatches.Patch(color='#534AB7',alpha=0.9,label='Batch A: ngram [1,2,3]'),
                   mpatches.Patch(color='#E24B4A',alpha=0.9,label='Batch B: ngram [2,3,4]'),
                   mpatches.Patch(color='#1D9E75',alpha=0.9,label='Terbaik')],fontsize=9)
ax.set_ylim(max(0,df_p2_v['f1_macro'].min()-0.05),min(1,df_p2_v['f1_macro'].max()+0.05))
sns.despine()
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/phase2_results.png', dpi=150, bbox_inches='tight')
plt.close()
print('  ✔ phase2_results.png tersimpan')

fig,axes=plt.subplots(2,2,figsize=(14,9)); axes=axes.flatten()
for idx,(col,lbl) in enumerate([('ngram','N-gram'),('filter_size','Filter'),
                                  ('dropout','Dropout'),('activation','Activation')]):
    g=df_p2_v.groupby(col)['f1_macro']; means=g.mean().reset_index(); stds=g.std().reset_index()
    xv=[str(v) for v in means[col]]
    axes[idx].bar(xv,means['f1_macro'],color='#534AB7',edgecolor='none',alpha=0.85,width=0.5)
    axes[idx].errorbar(xv,means['f1_macro'],yerr=stds['f1_macro'],fmt='none',ecolor='#888780',capsize=5)
    axes[idx].set_title(f'Dampak {lbl}',fontsize=11); axes[idx].set_xlabel(lbl); axes[idx].set_ylabel('F1-Macro')
    axes[idx].set_ylim(max(0,means['f1_macro'].min()-0.03),min(1,means['f1_macro'].max()+0.03))
    sns.despine(ax=axes[idx])
plt.suptitle('Phase 2 — Dampak Masing-masing Dimensi CNN Hyperparameter',fontsize=12,y=1.01)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/phase2_hyperparam_impact.png', dpi=150, bbox_inches='tight')
plt.close()
print('  ✔ phase2_hyperparam_impact.png tersimpan')

# ── Load p2a_epoch_history dari CSV jika tidak ada di memori ─────────────────
_p2a_hist_csv  = f'{OUTPUT_DIR}/phase2A_epoch_history.csv'
_p2a_fold_csv  = f'{OUTPUT_DIR}/phase2A_fold_details.csv'
_p2b_hist_csv  = f'{OUTPUT_DIR}/phase2B_epoch_history.csv'
_p2b_fold_csv  = f'{OUTPUT_DIR}/phase2B_fold_details.csv'

if 'p2a_epoch_history' not in dir() or not p2a_epoch_history:
    if os.path.exists(_p2a_hist_csv):
        p2a_epoch_history = pd.read_csv(_p2a_hist_csv).to_dict('records')
        print('✅ p2a_epoch_history dimuat dari CSV')
    else:
        p2a_epoch_history = []
        print('⚠️  phase2A_epoch_history.csv tidak ditemukan')

if 'p2a_fold_details' not in dir() or not p2a_fold_details:
    if os.path.exists(_p2a_fold_csv):
        p2a_fold_details = pd.read_csv(_p2a_fold_csv).to_dict('records')
        print('✅ p2a_fold_details dimuat dari CSV')
    else:
        p2a_fold_details = []
        print('⚠️  phase2A_fold_details.csv tidak ditemukan')

if 'p2b_epoch_history' not in dir() or not p2b_epoch_history:
    if os.path.exists(_p2b_hist_csv):
        p2b_epoch_history = pd.read_csv(_p2b_hist_csv).to_dict('records')
        print('✅ p2b_epoch_history dimuat dari CSV')
    else:
        p2b_epoch_history = []
        print('⚠️  phase2B_epoch_history.csv tidak ditemukan')

if 'p2b_fold_details' not in dir() or not p2b_fold_details:
    if os.path.exists(_p2b_fold_csv):
        p2b_fold_details = pd.read_csv(_p2b_fold_csv).to_dict('records')
        print('✅ p2b_fold_details dimuat dari CSV')
    else:
        p2b_fold_details = []
        print('⚠️  phase2B_fold_details.csv tidak ditemukan')

all_p2_hist  = p2a_epoch_history + p2b_epoch_history
all_p2_folds = p2a_fold_details  + p2b_fold_details

if 'best_p2' not in dir():
    best_p2 = df_p2_all.sort_values('f1_macro', ascending=False).iloc[0]
    print(f'✅ best_p2 dari df_p2_all: {best_p2["combo_label"]}')

plot_learning_curves(all_p2_hist, int(best_p2['combo_id']), best_p2['combo_label'],
                     f'{OUTPUT_DIR}/phase2_learning_curves_best.png')
plot_kfold_stability(all_p2_folds, 'Phase 2 A+B', f'{OUTPUT_DIR}/phase2_kfold_stability.png')
print('✔ Semua visualisasi Phase 2 tersimpan.')

---
# PHASE 3 — Validasi Kondisi Data & Evaluasi Final

**Prasyarat**: Semua `BEST_*` sudah diupdate.

| Kondisi | Teknik |
|---|---|
| S1 — Imbalanced | Class weighting |
| S2 — Balanced | Random undersampling |

> ⚠️ **Test set hanya digunakan SEKALI** di bagian Evaluasi Final.

In [ ]:
for name,val in [('BEST_LR_BERT',BEST_LR_BERT),('BEST_LR_CNN',BEST_LR_CNN),
                 ('BEST_BS',BEST_BS),('BEST_NGRAM',BEST_NGRAM),('BEST_FILTER',BEST_FILTER),
                 ('BEST_DROPOUT',BEST_DROPOUT),('BEST_ACTIVATION',BEST_ACTIVATION)]:
    assert val is not None,f'❌ {name} belum diisi!'
print('✅ Semua konfigurasi final tersedia.')
print(f'  Arsitektur : Dual-Path [CLS]+CNN → Concat → Dense({DENSE_SIZE}) → Dense({NUM_CLASSES})')
print(f'  LR         : lr_bert={BEST_LR_BERT} | lr_cnn={BEST_LR_CNN}')
print(f'  Training   : bs={BEST_BS} | wd={BEST_WD} (non-bias/LayerNorm only)')
print(f'  CNN        : ngram={BEST_NGRAM} | filter={BEST_FILTER} | dropout={BEST_DROPOUT} | act={BEST_ACTIVATION}')
print(f'  Fixed      : max_len={MAX_LEN}, warmup={WARMUP_RATIO}, eps={ADAM_EPS}, seed={SEED}')
print(f'               cls_dropout={CLS_DROPOUT}, dense_size={DENSE_SIZE}')

In [ ]:
p3_results=[]; p3_epoch_history=[]; p3_fold_details=[]
t_start_total=time.time()
conditions=[('S1','Imbalanced + Class Weighting'),('S2','Random Undersampling')]
cond_bar=tqdm(conditions,desc='Phase 3 K-Fold',bar_format='{l_bar}{bar:25}{r_bar}')
for cond,desc in cond_bar:
    cond_bar.set_description(f'Phase 3 [{cond}]')
    print(f'\n{"─"*65}\nKondisi {cond} — {desc}\n{"─"*65}')
    t_start=time.time()
    res=run_kfold(df_trainval,lr_bert=BEST_LR_BERT,lr_cnn=BEST_LR_CNN,
                  batch_size=BEST_BS,weight_decay=BEST_WD,
                  ngram=BEST_NGRAM,filter_size=BEST_FILTER,
                  dropout=BEST_DROPOUT,activation=BEST_ACTIVATION,
                  data_condition=cond,combo_id=cond,combo_label=f'best_{cond}')
    elapsed=time.time()-t_start
    p3_epoch_history.extend(res.pop('epoch_history'))
    p3_fold_details.extend(res.pop('fold_details'))
    p3_results.append({'condition':cond,'description':desc,**res})
    print(f'  └─ SELESAI | F1-Macro: {res["f1_macro"]:.4f} ± {res["f1_macro_std"]:.4f}')
    print(f'     Acc={res["accuracy"]:.4f} | Waktu: {elapsed/60:.1f} menit')

df_p3=pd.DataFrame(p3_results)
df_p3.to_csv(f'{OUTPUT_DIR}/phase3_kfold_results.csv',index=False)
pd.DataFrame(p3_epoch_history).to_csv(f'{OUTPUT_DIR}/phase3_epoch_history.csv',index=False)
pd.DataFrame(p3_fold_details).to_csv(f'{OUTPUT_DIR}/phase3_fold_details.csv',index=False)
total_time=time.time()-t_start_total
best_cond=df_p3.sort_values('f1_macro',ascending=False).iloc[0]['condition']
print(f'\n{"="*65}\n PHASE 3 K-FOLD — {total_time/60:.1f} menit\n{"="*65}')
print(df_p3[['condition','description','f1_macro','f1_macro_std','accuracy','precision','recall']].to_string(index=False))
print(f'\n Kondisi terbaik: {best_cond}')

In [ ]:
# ── Fallback: load best_cond dari CSV (aktif di setiap fase) ──────────────────
_p3_csv = f'{OUTPUT_DIR}/phase3_kfold_results.csv'
if os.path.exists(_p3_csv):
    _df_p3_tmp = pd.read_csv(_p3_csv)
    best_cond  = _df_p3_tmp.sort_values('f1_macro', ascending=False).iloc[0]['condition']
    print(f'✅ best_cond dimuat dari CSV: {best_cond}')
else:
    best_cond = None
    print('⚠️  phase3_kfold_results.csv belum ada — Phase 3 belum dijalankan.')

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(14,5))
mp=['f1_macro','f1_weighted','accuracy','precision','recall']
x,w=np.arange(len(mp)),0.35
s1=df_p3[df_p3['condition']=='S1'].iloc[0]
s2=df_p3[df_p3['condition']=='S2'].iloc[0]
axes[0].bar(x-w/2,[s1[m] for m in mp],w,color='#534AB7',alpha=0.9,label='S1',edgecolor='none')
axes[0].bar(x+w/2,[s2[m] for m in mp],w,color='#E24B4A',alpha=0.9,label='S2',edgecolor='none')
axes[0].set_xticks(x); axes[0].set_xticklabels(['F1-Macro','F1-Wgt','Accuracy','Precision','Recall'],fontsize=9)
axes[0].set_ylabel('Score'); axes[0].set_title('Komparasi Metrik S1 vs S2\n(Dual-Path [CLS]+CNN)')
axes[0].set_ylim(0,1.1); axes[0].legend(fontsize=10); sns.despine(ax=axes[0])

df_fd_p3=pd.DataFrame(p3_fold_details)
mb=['val_f1_macro','val_accuracy','val_precision','val_recall']
xp=np.arange(len(mb)); wb=0.3
for ci,(cv,cc) in enumerate([('S1','#534AB7'),('S2','#E24B4A')]):
    sub=df_fd_p3[df_fd_p3['condition']==cv]
    for mi,met in enumerate(mb):
        axes[1].boxplot(sub[met].values,positions=[xp[mi]+(ci-0.5)*wb],widths=wb*0.8,
                        patch_artist=True,boxprops=dict(facecolor=cc,alpha=0.7),
                        medianprops=dict(color='white',linewidth=2),
                        whiskerprops=dict(color=cc),capprops=dict(color=cc),
                        flierprops=dict(markerfacecolor=cc,marker='o',markersize=4))
axes[1].set_xticks(xp); axes[1].set_xticklabels(['F1-Macro','Accuracy','Precision','Recall'],fontsize=9)
axes[1].set_ylabel('Score per Fold'); axes[1].set_title('Stabilitas K-Fold: S1 vs S2')
axes[1].legend(handles=[mpatches.Patch(color='#534AB7',alpha=0.7,label='S1'),
                         mpatches.Patch(color='#E24B4A',alpha=0.7,label='S2')],fontsize=10)
sns.despine(ax=axes[1])
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/phase3_s1_vs_s2.png',dpi=150,bbox_inches='tight')
plt.close()
print('✔ Visualisasi Phase 3 K-Fold tersimpan.')

In [ ]:
# ── Fallback: load df_p3 dan p3_fold_details dari CSV (aktif di setiap fase) ──
_p3_kfold_csv = f'{OUTPUT_DIR}/phase3_kfold_results.csv'
_p3_fold_csv  = f'{OUTPUT_DIR}/phase3_fold_details.csv'

if 'df_p3' not in dir() or df_p3 is None:
    if os.path.exists(_p3_kfold_csv):
        df_p3 = pd.read_csv(_p3_kfold_csv)
        print('✅ df_p3 dimuat dari CSV')
    else:
        df_p3 = None
        print('⚠️  phase3_kfold_results.csv belum ada.')

if 'p3_fold_details' not in dir() or not p3_fold_details:
    if os.path.exists(_p3_fold_csv):
        p3_fold_details = pd.read_csv(_p3_fold_csv).to_dict('records')
        print('✅ p3_fold_details dimuat dari CSV')
    else:
        p3_fold_details = []
        print('⚠️  phase3_fold_details.csv belum ada.')

## Evaluasi Final pada Test Set

Model final dilatih pada seluruh train+val (5.313 data), dievaluasi pada **test set fixed 1.329 data** — satu kali saja.

In [ ]:
print(f'{"="*65}\n FINAL TRAINING & EVALUASI TEST SET (kondisi {best_cond})\n{"="*65}')
print(f' Arsitektur : Dual-Path [CLS]+CNN → Concat → Dense({DENSE_SIZE}) → Dense({NUM_CLASSES})')
print(f' LR         : lr_bert={BEST_LR_BERT} | lr_cnn={BEST_LR_CNN}')
print(f' CNN        : ngram={BEST_NGRAM} | filter={BEST_FILTER} | dropout={BEST_DROPOUT} | act={BEST_ACTIVATION}')
print(f' Training   : bs={BEST_BS} | wd={BEST_WD}')

X_full=df_trainval['text_bert'].values; y_full=df_trainval['label_id'].values
if best_cond=='S2': X_full,y_full=apply_undersampling(X_full,y_full)
criterion_f=build_criterion(y_full,best_cond,DEVICE)
# num_workers: 2 Colab | 0 Kaggle
train_dl_f=DataLoader(MBGDataset(X_full,y_full,tokenizer,MAX_LEN),
                      batch_size=BEST_BS,shuffle=True, num_workers=0)
test_dl_f =DataLoader(MBGDataset(df_test['text_bert'].values,df_test['label_id'].values,tokenizer,MAX_LEN),
                      batch_size=BEST_BS,shuffle=False,num_workers=0)
model_f    =IndoBERTCNN(BERT_MODEL,NUM_CLASSES,BEST_NGRAM,BEST_FILTER,
                        BEST_DROPOUT,BEST_ACTIVATION,CLS_DROPOUT,DENSE_SIZE).to(DEVICE)
optimizer_f=build_optimizer(model_f,BEST_LR_BERT,BEST_LR_CNN,BEST_WD)
total_steps=len(train_dl_f)*MAX_EPOCHS
scheduler_f=get_linear_schedule_with_warmup(optimizer_f,int(total_steps*WARMUP_RATIO),total_steps)

final_epoch_hist=[]; best_loss,patience_cnt,best_state=float('inf'),0,None
t_train=time.time()
epoch_bar=tqdm(range(1,MAX_EPOCHS+1),desc='Final Training',bar_format='{l_bar}{bar:25}{r_bar}')
for epoch in epoch_bar:
    tr_loss,_          =train_epoch(model_f,train_dl_f,optimizer_f,scheduler_f,criterion_f,DEVICE)
    val_loss,m,_,_,_   =eval_epoch(model_f,test_dl_f,criterion_f,DEVICE)
    final_epoch_hist.append({'epoch':epoch,'train_loss':round(tr_loss,4),
                              'val_loss':round(val_loss,4),**{f'val_{k}':v for k,v in m.items()}})
    epoch_bar.set_postfix({'tr':f'{tr_loss:.4f}','val':f'{val_loss:.4f}',
                           'f1':f'{m["f1_macro"]:.4f}','pat':f'{patience_cnt}/{PATIENCE}'})
    if val_loss<best_loss:
        best_loss=val_loss; best_state={k:v.clone() for k,v in model_f.state_dict().items()}; patience_cnt=0
    else:
        patience_cnt+=1
        if patience_cnt>=PATIENCE: print(f'  Early stopping ep{epoch}'); break

pd.DataFrame(final_epoch_hist).to_csv(f'{OUTPUT_DIR}/final_training_epoch_history.csv',index=False)
print(f' Training selesai — {(time.time()-t_train)/60:.1f} menit')

model_f.load_state_dict(best_state)
_,final_metrics,all_preds,all_labels,all_probs=eval_epoch(model_f,test_dl_f,criterion_f,DEVICE)
per_class=f1_score(all_labels,all_preds,average=None,zero_division=0)

print(f'\n{"="*65}\n EVALUASI FINAL — TEST SET ({best_cond})\n{"="*65}')
for k,v in final_metrics.items(): print(f'  {k:15s}: {v:.4f}')
print(f'\n{classification_report(all_labels,all_preds,target_names=LABEL_NAMES)}')
print(' Per-class F1: '+' | '.join([f'{LABEL_NAMES[i]}={per_class[i]:.4f}' for i in range(NUM_CLASSES)]))

plot_confusion_matrix_pair(
    all_labels,all_preds,
    title_suffix=f'IndoBERT+CNN Dual-Path ({best_cond}) | F1={final_metrics["f1_macro"]:.4f}',
    save_prefix=f'{OUTPUT_DIR}/proposed_{best_cond}'
)

df_pred=df_test[['full_text','text_bert','label']].copy() if 'full_text' in df_test.columns \
        else df_test[['text_bert','label']].copy()
df_pred['predicted']    =[ID2LABEL[p] for p in all_preds]
df_pred['correct']      =df_pred['label']==df_pred['predicted']
df_pred['confidence']   =np.round(np.max(all_probs,axis=1),4)
df_pred['prob_positive']=np.round(all_probs[:,0],4)
df_pred['prob_negative']=np.round(all_probs[:,1],4)
df_pred['prob_neutral'] =np.round(all_probs[:,2],4)
df_pred.to_csv(f'{OUTPUT_DIR}/test_predictions.csv',index=False,encoding='utf-8-sig')
print(f'\n ✔ test_predictions.csv ({len(df_pred):,} baris) | Accuracy: {df_pred["correct"].mean()*100:.2f}%')

model_path=f'{MODEL_DIR}/indobert_cnn_dualpath_{best_cond}.pt'
torch.save({
    'model_state_dict': model_f.state_dict(),
    'architecture'    : 'Dual-Path [CLS] + CNN 1D',
    'lr_config'       : {'lr_bert':BEST_LR_BERT,'lr_cnn':BEST_LR_CNN,
                         'batch_size':BEST_BS,'weight_decay':BEST_WD,
                         'strategy':'differential_lr_weight_decay_exclusion'},
    'cnn_config'      : {'ngram':BEST_NGRAM,'filter':BEST_FILTER,
                         'dropout':BEST_DROPOUT,'activation':BEST_ACTIVATION},
    'fixed_config'    : {'max_len':MAX_LEN,'warmup_ratio':WARMUP_RATIO,
                         'adam_eps':ADAM_EPS,'seed':SEED,
                         'cls_dropout':CLS_DROPOUT,'dense_size':DENSE_SIZE,
                         'no_decay':['bias','LayerNorm.weight','LayerNorm.bias'],
                         'pooling':'GlobalMaxPool'},
    'data_condition'  : best_cond,
    'test_metrics'    : final_metrics,
    'per_class_f1'    : {LABEL_NAMES[i]:round(per_class[i],4) for i in range(NUM_CLASSES)},
    'label2id':LABEL2ID,'id2label':ID2LABEL,'bert_model_name':BERT_MODEL,
},model_path)

print(f'\n{"="*65}\n RINGKASAN OUTPUT NOTEBOOK 03\n{"="*65}')
outputs=[
    ('phase1_results.csv','Metrik per kombinasi Phase 1'),
    ('phase1_epoch_history.csv','Epoch history semua kombinasi Phase 1'),
    ('phase1_fold_details.csv','Fold details semua kombinasi Phase 1'),
    ('phase1_summary_top10.csv','Top-10 Phase 1 — siap laporan'),
    ('phase1_results.png','Heatmap + bar top-10'),
    ('phase1_lr_sensitivity.png','Line plot sensitivitas LR'),
    ('phase1_learning_curves_best.png','Learning curve config terbaik Phase 1'),
    ('phase1_kfold_stability.png','Stabilitas K-Fold Phase 1'),
    ('phase2A_results.csv','Metrik per kombinasi Phase 2A'),
    ('phase2B_results.csv','Metrik per kombinasi Phase 2B'),
    ('phase2_combined_results.csv','Gabungan Phase 2A + 2B'),
    ('phase2_summary_top10.csv','Top-10 Phase 2 — siap laporan'),
    ('phase2_results.png','Bar chart A+B dengan warna batch'),
    ('phase2_hyperparam_impact.png','Dampak per dimensi CNN'),
    ('phase2_learning_curves_best.png','Learning curve config terbaik Phase 2'),
    ('phase2_kfold_stability.png','Stabilitas K-Fold Phase 2'),
    ('phase3_epoch_history.csv', 'Epoch history Phase 3 K-Fold'),
    ('phase3_kfold_results.csv','Summary K-Fold S1 vs S2'),
    ('phase3_fold_details.csv','Fold details S1 dan S2'),
    ('phase3_s1_vs_s2.png','Grouped bar + box plot S1 vs S2'),
    ('final_training_epoch_history.csv','Epoch history final training'),
    (f'proposed_{best_cond}_cm_pair.png','Confusion matrix unnormalized + normalized'),
    ('test_predictions.csv','Prediksi per tweet + confidence + prob per kelas'),
    (f'indobert_cnn_dualpath_{best_cond}.pt','Model checkpoint final (Dual-Path)'),
]
for fname,desc in outputs:
    path=f'{OUTPUT_DIR}/{fname}' if not fname.endswith('.pt') else f'{MODEL_DIR}/{fname}'
    e='✔' if os.path.exists(path) else '○'
    print(f'  {e} {fname:<50} {desc}')
print(f'\n Lanjut ke Notebook 04 — BERT-only Baseline')
print(f' Gunakan: lr_bert={BEST_LR_BERT} | bs={BEST_BS} | wd={BEST_WD}')

---

## Catatan Metodologis untuk Laporan

### Arsitektur Dual-Path [CLS] + CNN
Arsitektur Dual-Path menggabungkan dua representasi komplementer dari output IndoBERT:

**Path 1 — [CLS] token** mengandung representasi konteks global seluruh kalimat hasil agregasi self-attention 12 layer. Dirancang oleh Devlin et al. (2019) untuk task klasifikasi. Dropout kecil (0.1) diterapkan sebelum concatenate.

**Path 2 — CNN 1D multi-kernel** menangkap pola n-gram lokal secara paralel pada berbagai ukuran window. Frasa sentimen seperti *"tidak bagus"*, *"sangat buruk"*, *"cukup baik"* ditangkap melalui GlobalMaxPool dari setiap kernel.

**Concatenate** menggabungkan keduanya sebelum classifier. CNN **melengkapi** [CLS], tidak bersaing dengannya. Dimensi input classifier: 768 + filter_size × n_kernels.

### Differential Learning Rate
IndoBERT (pre-trained) menggunakan `lr_bert` kecil untuk menghindari *catastrophic forgetting*. CNN + head (from scratch) menggunakan `lr_cnn` lebih besar.  
**Ref**: Howard & Ruder (2018); Sun et al. (2019); Devlin et al. (2019).

### Weight Decay Exclusion
`bias` dan `LayerNorm.*` dikecualikan dari weight decay (`wd=0.0`) pada semua komponen.  
**Ref**: Loshchilov & Hutter (2019).

### Evaluasi
Metrik utama: **F1-Macro**. Test set (20%) dipisah sebelum training dan digunakan **sekali** pada evaluasi final.